In [ ]:
# 1. 구글드라이브 연동 및 깃허브 클론/풀 (동일한 환경 세팅)
import os
import sys
import shutil
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if os.path.exists('/content/korean-chatbot'):
    %cd korean-chatbot
    !git pull
else:
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install -r requirements.txt




In [ ]:

# 2. 필요한 파일들 로드 (vocab.json 및 구글 드라이브에서 model.pt 가져오기)
import shutil, os

os.makedirs("models", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/models/vocab.json", "models/vocab.json")

# 구글 드라이브에 저장되어 있는 학습 완료된 model.pt 파일을 복사해옵니다.
drive_model_path = "/content/drive/MyDrive/korean-chatbot/models/model.pt"
local_model_path = "models/model.pt"

if os.path.exists(drive_model_path):
    shutil.copy(drive_model_path, local_model_path)
    print("✅ 구글 드라이브로부터 model.pt 로드 완료!")
else:
    print(f"❌ 구글 드라이브 경로에 모델 파일이 없습니다: {drive_model_path}")

In [ ]:


# 3. GPU 장치 정의 및 A100 가속 활성화
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

if torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("🚀 A100 GPU 감지: TF32 매트릭스 연산 가속을 켭니다.")

In [ ]:
# ==========================================
# 1. 구글드라이브 연동 및 패키지 설치 + 파일 로드
# ==========================================
# (이전과 동일하게 드라이브 마운트 및 git pull, model.pt 복사 진행...)

# ==========================================
# 2. 깃허브 src 폴더에서 클래스 직접 import 하기
# ==========================================
# ※ 만약 src 폴더가 파이썬 경로에 안 잡히면 아래 두 줄을 실행해줍니다.
import sys
sys.path.append('/content/korean-chatbot/src') 

# 레포지토리 내부 파일에서 직접 클래스를 가져옵니다.
# (실제 파일명과 클래스명에 맞게 import 문을 수정해주세요)
from model import Transformer
from tokenizer import BPETokenizer

import torch
import os

# ==========================================
# 3. 장치 설정 및 객체 생성
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
local_vocab_path = "models/vocab.json"
local_model_path = "models/model.pt"

# 토크나이저 로드
tok = BPETokenizer()
tok.load(local_vocab_path)
vocab_size = tok.tokenizer.get_vocab_size()

# 깃허브에서 가져온 클래스로 빈 뼈대 모델 생성
model = Transformer(vocab_size=vocab_size)

# 이미 가지고 계신 model.pt 가중치 주입
if os.path.exists(local_model_path):
    model.load_state_dict(torch.load(local_model_path, map_location=device))
    model = model.to(device)
    model.eval()
    print("✨ 깃허브 src 코드를 활용해 모델 로드 완료!")

# ==========================================
# 4. 추론 테스트 실행
# ==========================================
# (이후 test_inputs 돌리는 generate 루프 실행...)

In [ ]:

# 5. 테스트 입력 및 문장 생성 (Inference 실행)
test_inputs = [
    # 인사
    "안녕하세요",
    "반갑습니다",
    "좋은 아침이에요",
    
    # 한국 역사
    "조선은",
    "한국의 역사는",
    "고려시대에는",
    "삼국시대란",
    
    # 지식
    "인공지능이란",
    "머신러닝은",
    "딥러닝의 원리는",
    "트랜스포머 모델은",
    
    # 나무위키 스타일
    "대한민국은",
    "서울은",
    "한국어란",
    "김치는",
    
    # 문장 완성
    "오늘 날씨가",
    "밥을 먹으러",
    "학교에서",
    "회사에서",
]

print("\n🤖 문장 생성을 시작합니다...\n")
print("=" * 50)

# 가속화된 추론 환경 구축을 위한 컨텍스트 매니저 설정
with torch.no_grad():
    # 학습 때와 동일하게 bfloat16 AMP 가속을 활용해 추론 속도를 높입니다.
    with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32):
        for text in test_inputs:
            result = model.generate(text, tok)
            print(f"입력: {text}")
            print(f"출력: {result}")
            print("-" * 50)

print("\n🎉 모든 추론이 완료되었습니다!")

In [ ]:
import torch

# 대화 루프 시작 전 모델을 확실하게 평가(추론) 모드로 설정합니다.
model.eval()

print("💬 챗봇 대화 루프를 시작합니다. 문장을 입력하고 엔터를 누르세요.")
print("종료하려면 'q'를 입력하세요.\n")

while True:
    # 1. 사용자 입력 받기
    prompt = input("입력 ('q'로 종료): ")
    
    # 2. 종료 조건 체크
    if prompt.strip() == 'q':
        print("👋 대화를 종료합니다.")
        break
        
    # 빈 입력 방지 예외 처리
    if not prompt.strip():
        print("⚠️ 내용을 입력해주세요.\n")
        continue
        
    # 3. 모델 가속 및 문장 생성 (Inference)
    with torch.no_grad():
        with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32):
            # 기존 Transformer 클래스의 generate 정의에 맞게 인자명(max_length)을 매칭합니다.
            result = model.generate(prompt, tok, max_length=100)
            
    # 4. 결과 출력
    print(f"출력: {result}\n")
    print("-" * 50)